# 03 — Iterative Debugging Loop & Execution Reward Design
**Goal**: Implement the multi-turn agentic debugging loop ($K=3$ turns) with error feedback (Verification Check V2), define execution-guided reward functions, synthetic bug injection, and collect trajectory preference pairs for DPO.

---

## Step 1: Environment & Module Setup

In [ ]:
import sys, os, shutil

# 1. Clear any previously cached 'src' modules from Python memory
for mod_name in list(sys.modules.keys()):
    if mod_name == "src" or mod_name.startswith("src."):
        del sys.modules[mod_name]

# 2. Prepare working copy of 'src' in /kaggle/working
input_src = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "src" in dirs:
        input_src = os.path.join(root, "src")
        break

working_src = "/kaggle/working/src"
if input_src:
    if os.path.exists(working_src):
        shutil.rmtree(working_src)
    shutil.copytree(input_src, working_src)
    print(f"SUCCESS: Prepared working copy of 'src' at {working_src}")

# 3. Patch debugging/debug_loop.py with full implementations & initial_code support
debug_file = os.path.join(working_src, "debugging", "debug_loop.py")
debug_clean_code = '''"""Iterative execution-feedback debugging loop."""

import random
import re
import torch
from typing import Any, Callable, Dict, List, Optional

from src.execution.executor import ExecutionResult, PythonSandbox, run_code
from src.execution.status import ExecutionStatus
from src.models.generation import extract_code_block
from src.rewards.execution_reward import compute_partial_reward


def build_prompt(problem: str, prev_code: str = None, traceback: str = None) -> str:
    if prev_code is None:
        return f"### Problem:\n{problem}\n\n### Write a Python solution:\n```python"
    else:
        tb_trimmed = (traceback or "")[:512]
        return (
            f"### Problem:\n{problem}\n\n"
            f"### Your previous code:\n```python\n{prev_code}\n```\n\n"
            f"### Error you received:\n{tb_trimmed}\n\n"
            f"### Fixed version:\n```python"
        )


def agentic_debug_loop(model, tokenizer, problem: str, test_cases: list = None, K: int = 3, initial_code: str = None) -> list:
    history = []
    code = initial_code
    traceback = ""
    device = next(model.parameters()).device
    test_cases = test_cases or []

    for turn in range(K):
        if turn == 0 and code is not None:
            prompt = build_prompt(problem)
        else:
            prompt = build_prompt(problem, code, traceback)
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.2 if turn == 0 else 1.0,
                    do_sample=(turn > 0),
                    top_p=0.95,
                    pad_token_id=tokenizer.pad_token_id,
                )
            full = tokenizer.decode(output[0], skip_special_tokens=True)
            code = extract_code_block(full, prompt)

        full_code = code + "\n\n" + "\n".join(test_cases) if test_cases else code
        result = run_code(full_code)
        history.append({"turn": turn + 1, "prompt": prompt, "code": code, "result": result})
        if result["status"] == "AC":
            break
        traceback = result.get("traceback", "") or result.get("output", "")

    return history


def agentic_loop_no_feedback(model, tokenizer, problem: str, test_cases: list = None, K: int = 3) -> list:
    history = []
    code = None
    fake_errors = ['NameError: name "x" is not defined', 'IndexError: list index out of range', 'TypeError: unsupported operand type(s)']
    device = next(model.parameters()).device
    test_cases = test_cases or []
    for turn in range(K):
        fake_tb = random.choice(fake_errors)
        prompt = build_prompt(problem, code, fake_tb)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=512, temperature=0.2 if turn == 0 else 1.0, do_sample=(turn > 0), top_p=0.95, pad_token_id=tokenizer.pad_token_id)
        full = tokenizer.decode(output[0], skip_special_tokens=True)
        code = extract_code_block(full, prompt)
        full_code = code + "\n\n" + "\n".join(test_cases) if test_cases else code
        result = run_code(full_code)
        history.append({"turn": turn + 1, "code": code, "result": result})
        if result["status"] == "AC":
            break
    return history


def format_execution_feedback(result: ExecutionResult) -> str:
    if result.status == ExecutionStatus.AC:
        return "All test cases PASSED successfully."
    feedback = [f"Execution Status: {result.status}"]
    if result.traceback:
        feedback.append(f"Traceback / Error Details:\n{result.traceback}")
    elif result.stderr:
        feedback.append(f"Error Output (stderr):\n{result.stderr}")
    elif result.stdout:
        feedback.append(f"Output (stdout):\n{result.stdout}")
    if result.total_tests > 0:
        feedback.append(f"Passed {result.passed_tests} / {result.total_tests} test cases.")
    return "\n".join(feedback)


class DebugLoop:
    def __init__(self, sandbox: Optional[PythonSandbox] = None, max_turns: int = 5, traceback_token_cap: int = 500):
        self.sandbox = sandbox or PythonSandbox()
        self.max_turns = max_turns
        self.traceback_token_cap = traceback_token_cap

    def build_initial_prompt(self, problem_description: str) -> str:
        return f"Solve the following programming problem in Python.\nProblem Statement:\n{problem_description}\n\nWrite clean, self-contained Python code wrapped in ```python ... ```."

    def build_turn_prompt(self, problem_description: str, previous_code: str, execution_result: ExecutionResult) -> str:
        feedback = format_execution_feedback(execution_result)
        return f"Problem Statement:\n{problem_description}\n\nPrevious solution failed execution.\nPrevious Code:\n```python\n{previous_code}\n```\n\nExecution Feedback:\n{feedback}\n\nIdentify the bug, correct the code, and provide the fixed Python solution wrapped in ```python ... ```."

    def run_session(self, problem_description: str, test_cases: List[Dict[str, Any]], model_fn: Callable[[str], str], initial_code: Optional[str] = None) -> Dict[str, Any]:
        history = []
        solved = False
        solved_turn = None
        current_code = initial_code
        for turn_idx in range(1, self.max_turns + 1):
            if current_code is None:
                prompt = self.build_initial_prompt(problem_description)
                raw_response = model_fn(prompt)
                current_code = extract_code_block(raw_response)
            else:
                prompt = ""
            exec_res = self.sandbox.run_tests(current_code, test_cases)
            reward = compute_partial_reward(exec_res)
            turn_info = {"turn": turn_idx, "prompt": prompt, "code": current_code, "execution_result": exec_res.to_dict(), "reward": reward, "status": exec_res.status}
            history.append(turn_info)
            if exec_res.status == ExecutionStatus.AC:
                solved = True
                solved_turn = turn_idx
                break
            if turn_idx < self.max_turns:
                turn_prompt = self.build_turn_prompt(problem_description, current_code, exec_res)
                raw_response = model_fn(turn_prompt)
                current_code = extract_code_block(raw_response)
        return {"solved": solved, "solved_turn": solved_turn, "total_turns": len(history), "max_turns": self.max_turns, "history": history, "final_status": history[-1]["status"], "final_reward": history[-1]["reward"]}
'''
with open(debug_file, "w", encoding="utf-8") as f:
    f.write(debug_clean_code)

# 4. Patch models/loader.py with dual torchao override right before get_peft_model
loader_file = os.path.join(working_src, "models", "loader.py")
loader_clean_code = '''"""Model loading utilities with quantization and LoRA configuration."""

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEFAULT_MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"


def load_model_and_tokenizer(
    model_name: str = DEFAULT_MODEL_NAME,
    load_in_4bit: bool = True,
    lora_r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.05,
    attach_lora: bool = True,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    try:
        import bitsandbytes
        has_bnb = True
    except (ImportError, Exception):
        has_bnb = False

    if load_in_4bit and has_bnb and torch.cuda.is_available():
        try:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            device_map = "auto"
        except Exception:
            bnb_config = None
            device_map = "auto" if torch.cuda.is_available() else None
    else:
        bnb_config = None
        device_map = "auto" if torch.cuda.is_available() else None

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map=device_map,
        trust_remote_code=True,
    )

    if attach_lora:
        try:
            import peft.import_utils
            peft.import_utils.is_torchao_available = lambda: False
        except Exception:
            pass
        try:
            import peft.tuners.lora.torchao
            peft.tuners.lora.torchao.is_torchao_available = lambda: False
        except Exception:
            pass

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=lora_dropout,
            task_type=TaskType.CAUSAL_LM,
        )
        model = get_peft_model(model, lora_config)

    return model, tokenizer
'''
with open(loader_file, "w", encoding="utf-8") as f:
    f.write(loader_clean_code)

# 5. Patch training/dpo.py to prevent top-level TRL import error
dpo_file = os.path.join(working_src, "training", "dpo.py")
dpo_clean_code = '''"""DPO training entry points."""

from datasets import Dataset
from src.debugging.debug_loop import agentic_debug_loop


def make_preference_pairs(problems, model, tokenizer, K: int = 3) -> Dataset:
    pairs = []
    for prob in problems:
        question = prob.get("question", prob.get("prompt", ""))
        test_cases = [prob["test"] + f"\ncheck({prob["entry_point"]})\n"] if "test" in prob and "entry_point" in prob else []
        history = agentic_debug_loop(model, tokenizer, question, test_cases, K=K)

        ac_turns = [h for h in history if h["result"]["status"] == "AC"]
        bad_turns = [h for h in history if h["result"]["status"] in ("CE", "RE", "WA", "TLE", "MLE")]

        if ac_turns and bad_turns:
            pairs.append({
                "prompt": question,
                "chosen": ac_turns[0]["code"],
                "rejected": bad_turns[0]["code"],
            })

    return Dataset.from_list(pairs)


def run_dpo_training(model, tokenizer, preference_data, output_dir="./checkpoints/dpo", beta=0.1, learning_rate=5e-5, num_train_epochs=3, per_device_train_batch_size=4):
    try:
        from trl import DPOConfig, DPOTrainer
    except ImportError as e:
        raise ImportError(f"TRL library is required for DPO training: {e}")

    dpo_config = DPOConfig(
        beta=beta,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        output_dir=output_dir,
        fp16=True,
        report_to="none",
    )

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_config,
        train_dataset=preference_data,
        tokenizer=tokenizer,
    )

    dpo_trainer.train()
    dpo_trainer.save_model(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")
    return dpo_trainer
'''
with open(dpo_file, "w", encoding="utf-8") as f:
    f.write(dpo_clean_code)

# 6. Set /kaggle/working as top priority path
sys.path.insert(0, "/kaggle/working")

# 7. Import all modules
import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.debugging.debug_loop import build_prompt, agentic_debug_loop, DebugLoop
from src.rewards.execution_reward import compute_reward, compute_reward_binary, compute_partial_reward, compute_status_aware_reward
from src.training.dpo import make_preference_pairs
from src.error_injection import BugInjector, BugType

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment & modules patched and initialized successfully!")

## Step 2: Synthetic Bug Injection Demo

In [ ]:
injector = BugInjector(seed=42)
correct_code = "def multiply(a, b):\n    return a * b"

print("=== Synthetic Bug Injection ===")
for category in list(BugType):
    res = injector.inject_bug(correct_code, category=category)
    print(f"[{res['bug_type'].upper()}] -> {res['description']}")

## Step 3A: Controlled Bug-Injection Self-Correction (Verification Check V2 — Part A)
Injects a synthetic logic bug into Turn 1 code to produce a guaranteed `WA`/`RE` failure $\rightarrow$ captures traceback $\rightarrow$ model receives execution feedback in Turn 2 and generates the fix to reach `AC`.

In [ ]:
!pip uninstall -y torchao

import sys, os
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
print(f"Loading model and tokenizer: {MODEL_NAME} in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    load_in_4bit=False,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    attach_lora=True,
)

humaneval = load_dataset('openai_humaneval', split='test')
prob0 = humaneval[0]
problem_stmt = prob0['prompt']
test_cases0 = [prob0['test'] + f"\ncheck({prob0['entry_point']})\n"]

# 1. Create a controlled buggy solution for Turn 1
correct_ref = """from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
"""

injector = BugInjector(seed=42)
bug_res = injector.inject_bug(correct_ref, category=BugType.LOGIC)
buggy_turn1_code = bug_res['buggy_code']

print("=== Verification Check V2 (Part A: Controlled Self-Correction Trajectory) ===")
print(f"Injected Bug Description: {bug_res['description']}")

trajectory_a = agentic_debug_loop(model, tokenizer, problem_stmt, test_cases=test_cases0, initial_code=buggy_turn1_code, K=2)

for turn_info in trajectory_a:
    print(f"\n--- [Turn {turn_info['turn']}] ---")
    print(f"Status: {turn_info['result']['status']}")
    print(f"Code Snippet:\n{turn_info['code'][:180]}...")
    if turn_info['result'].get('traceback'):
        print(f"Feedback / Traceback:\n{turn_info['result']['traceback'][:200]}...")

print("\nVerification Check V2 Part A completed!")

## Step 3B: Natural Zero-Shot Failure Self-Correction (Verification Check V2 — Part B)
Evaluates a problem where model zero-shot generation naturally fails on Turn 1 $\rightarrow$ receives test failure feedback $\rightarrow$ self-corrects on Turn 2 to reach `AC`.

In [ ]:
# Problem 10 in HumanEval requires specific string palindrome logic that can fail zero-shot
prob10 = humaneval[10]
problem_b_stmt = prob10['prompt']
test_cases10 = [prob10['test'] + f"\ncheck({prob10['entry_point']})\n"]

print("=== Verification Check V2 (Part B: Natural Failure Trajectory) ===")
print(f"Problem: {prob10['entry_point']}")

trajectory_b = agentic_debug_loop(model, tokenizer, problem_b_stmt, test_cases=test_cases10, K=3)

for turn_info in trajectory_b:
    print(f"\n--- [Turn {turn_info['turn']}] ---")
    print(f"Status: {turn_info['result']['status']}")
    print(f"Code Snippet:\n{turn_info['code'][:180]}...")
    if turn_info['result'].get('traceback'):
        print(f"Feedback / Traceback:\n{turn_info['result']['traceback'][:200]}...")

print("\nVerification Check V2 Part B completed!")

## Step 4: Execution Reward Function Verification
Validates reward scores for AC ($+1.0$), WA/partial pass ($0.0 - 1.0$), and CE/RE ($-0.2$ penalty).

In [ ]:
print("--- Dense Reward Calculations ---")
print(f"AC (5/5 tests): {compute_reward('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward('WA', 3, 5)}")
print(f"CE (0/5 tests): {compute_reward('CE', 0, 5)}")
print(f"RE (0/5 tests): {compute_reward('RE', 0, 5)}")

print("\n--- Binary Reward Calculations (RQ4 Ablation) ---")
print(f"AC (5/5 tests): {compute_reward_binary('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward_binary('WA', 3, 5)}")

## Step 5: Preference Pair Generation for DPO
Collects `(prompt, chosen, rejected)` trajectory pairs from debug rollouts for DPO training.

In [ ]:
print("Generating sample DPO preference pairs...")

eval_data = load_dataset('openai_humaneval', split='test[:5]')
pref_data = make_preference_pairs(eval_data, model, tokenizer, K=3)
print(f"Total preference pairs collected: {len(pref_data)}")
if len(pref_data) > 0:
    print(f"Sample Chosen:\n{pref_data[0]['chosen'][:100]}...")
    print(f"Sample Rejected:\n{pref_data[0]['rejected'][:100]}...")